## Modelo datos estructurados
Este modelo parte del original de Tabulares trabajado en clase

In [1]:
#Import de librerias basicas tablas y matrices
import numpy as np 
import pandas as pd 

#Gradient Boosting
import lightgbm as lgb

#Funciones auxiliares sklearn
from sklearn.model_selection import train_test_split, StratifiedKFold #Split y cross Validation
from sklearn.metrics import cohen_kappa_score, accuracy_score, balanced_accuracy_score #Metricas
from sklearn.utils import shuffle 

#Visualizacióon
from plotly import express as px

#Plot de matriz de confusion normalizada en actuals
from utils import plot_confusion_matrix

import os

#Optimizacion de hiperparametros
import optuna
from optuna.artifacts import FileSystemArtifactStore, upload_artifact

#Guardado de objetos en archivos joblib
from joblib import load, dump

# textblob
from textblob import TextBlob


c:\Users\josek\miniconda3\envs\ldi2_cuda\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Paths para acceso archivos
#Este notebook asume la siguiente estructura de carpetas a partir de la ubicacion de base_dir 
#(dos niveles arriba de la cƒarpeta donde se ejecuta el notebook). 
# /ƒ/ƒ
# /UA_MDM_Labo2/inputƒ
# /UA_MDM_Labo2/input/petfinder-adoption-prediction/            <- Aca deben ir todos los archivos de datos de la competencia 
# /UA_MDM_Labo2/tutoriales/                       <- Aca deben poner los notebooks y scripts que les compartimos
# /UA_MDM_Labo2/work/                             <- Resultados de notebooks iran dentro de esta carpeta en subcarpetas
# /UA_MDM_Labo2/work/models/                     <- Modelos entrenados en archivos joblibs
# /UA_MDM_Labo2/work/optuna_temp_artifacts/      <- Archivos que queremos dejar como artefacto de un trial de optuna (optuna los copiara a la carpeta de abajo)
# /UA_MDM_Labo2/work/optuna_artifacts/           <- Archivos con artefactos que sibimos a optuna

#Subimos dos niveles para quedar en la carpeta que contiene input y UA_MDM_Labo2
BASE_DIR = '../'

#Datos de entrenamiento 
PATH_TO_TRAIN = os.path.join(BASE_DIR, "input/petfinder-adoption-prediction/train/train.csv")

#Salida de modelos entrenados
PATH_TO_MODELS = os.path.join(BASE_DIR, "work/models")

#Artefactos a subir a optuna
PATH_TO_TEMP_FILES = os.path.join(BASE_DIR, "work/optuna_temp_artifacts")

#Artefactos que optuna gestiona
PATH_TO_OPTUNA_ARTIFACTS = os.path.join(BASE_DIR, "work/optuna_artifacts")


SEED = 42 #Semilla de procesos aleatorios (para poder replicar exactamente al volver a correr un modelo)
TEST_SIZE = 0.2 #Facción para train/test= split

In [3]:
# Datos Tabulares
dataset = pd.read_csv(PATH_TO_TRAIN)

In [ ]:
# Feature Engineering

# link: https://medium.com/%40thom42/pet-cat-and-dog-adoption-speed-prediction-kaggle-petfindermy-part-1-eda-631f9538cde6
dataset['HasName'] = dataset['Name'].apply(
    lambda x: 0 if pd.isna(x) or str(x).strip().lower() in ['not named', 'no name yet', 'none', 'unnamed', 'no name'] else 1
)


## len descripcion

dataset['DescLen'] = dataset['Description'].apply(
    lambda x: len(str(x))
)


## Si cumple todos los requisitos de salud

dataset['score_salud'] = ( 
    (dataset['Vaccinated'] == 1).astype(int) + 
    (dataset['Sterilized'] == 1).astype(int) + 
    (dataset['Dewormed'] == 1).astype(int) 
)

## Pureza
dataset["Puro"] = np.where((dataset["Breed2"] > 0) | (dataset["Breed1"] == 307), 0, 1)


## cachorro | inter | adulto

def age_range(row):
    if row.Age < 6:
        return 1
    elif row.Age < 24:
        return 2
    else:
        return 3
        
dataset["AgeRange"] = dataset.apply(age_range, axis=1)


## Cantidad de colores

def color_checker(row):
    if row.Color3 != 0:
        return 3
    else:
        if row.Color2 != 0:
            return 2
        else:
            return 1
        
dataset["Color_comb"] = dataset.apply(color_checker, axis=1)

# Tiene algo de contenido

def content_checker(row):
    cant = row.VideoAmt + row.PhotoAmt
    if cant > 0:
        return 1
    else:
        return 0
        
dataset["Content"] = dataset.apply(content_checker, axis=1)

# ln Fee
dataset['logFee'] = dataset['Fee'].apply(lambda x: np.log(x))

# Analisis de sentimiento
def getPolarity(row):
    text = str(row.Description)
    return TextBlob(text).sentiment.polarity
    
dataset["polarity"] = dataset.apply(getPolarity, axis = 1)

def getSubjectivity(row):
    text = str(row.Description)
    return TextBlob(text).sentiment.subjectivity

dataset["Subjectivity"] = dataset.apply(getSubjectivity, axis = 1)

# Estados https://www.kaggle.com/c/petfinder-adoption-prediction/discussion/78040

# state GDP: https://en.wikipedia.org/wiki/List_of_Malaysian_states_by_GDP in billions
state_gdp = {
    41336: 116.679,
    41325: 40.596,
    41367: 23.02,
    41401: 190.075,
    41415: 5.984,
    41324: 37.274,
    41332: 42.389,
    41335: 52.452,
    41330: 67.629,
    41380: 5.642,
    41327: 81.284,
    41345: 80.167,
    41342: 121.414,
    41326: 280.698,
    41361: 32.270
}

# state population: https://en.wikipedia.org/wiki/Malaysia
state_population = {
    41336: 33.48283,
    41325: 19.47651,
    41367: 15.39601,
    41401: 16.74621,
    41415: 0.86908,
    41324: 8.21110,
    41332: 10.21064,
    41335: 15.00817,
    41330: 23.52743,
    41380: 2.31541,
    41327: 15.61383,
    41345: 32.06742,
    41342: 24.71140,
    41326: 54.62141,
    41361: 10.35977
}

dataset["state_gdp"] = dataset['State'].map(state_gdp)
dataset["state_population"] = dataset['State'].map(state_population)
dataset["gdp_vs_population"] = dataset["state_gdp"] / dataset["state_population"]

C:\Users\josek\AppData\Local\Temp\ipykernel_22388\2836938963.py:66: RuntimeWarning: divide by zero encountered in log
  dataset['logFee'] = dataset['Fee'].apply(lambda x: np.log(x))


In [5]:
#Columnas del dataset
dataset.columns

Index(['Type', 'Name', 'Age', 'Breed1', 'Breed2', 'Gender', 'Color1', 'Color2',
       'Color3', 'MaturitySize', 'FurLength', 'Vaccinated', 'Dewormed',
       'Sterilized', 'Health', 'Quantity', 'Fee', 'State', 'RescuerID',
       'VideoAmt', 'Description', 'PetID', 'PhotoAmt', 'AdoptionSpeed',
       'HasName', 'DescLen', 'score_salud', 'Puro', 'AgeRange', 'Color_comb',
       'Content', 'logFee', 'polarity', 'Subjectivity', 'state_gdp',
       'state_population', 'gdp_vs_population'],
      dtype='object')

In [ ]:
#Guardo el dataset para reutilizar en otros modelos
dataset.to_csv("train_FE.csv", index= False) #ToDo poner bien el path

In [8]:
#Separo un 20% para test estratificado opr target
train, test = train_test_split(dataset,
                               test_size = TEST_SIZE,
                               random_state = SEED,
                               stratify = dataset.AdoptionSpeed)

In [7]:
#Armo listas con features de texto y numericas
char_feats = [f for f in dataset.columns if dataset[f].dtype=='O']
numeric_feats = [f for f in dataset.columns if dataset[f].dtype!='O']

In [8]:
char_feats

['Name', 'RescuerID', 'Description', 'PetID']

In [9]:
#Lista de features numericas
numeric_feats

['Type',
 'Age',
 'Breed1',
 'Breed2',
 'Gender',
 'Color1',
 'Color2',
 'Color3',
 'MaturitySize',
 'FurLength',
 'Vaccinated',
 'Dewormed',
 'Sterilized',
 'Health',
 'Quantity',
 'Fee',
 'State',
 'VideoAmt',
 'PhotoAmt',
 'AdoptionSpeed',
 'HasName',
 'DescLen',
 'score_salud',
 'Puro',
 'AgeRange',
 'Color_comb',
 'Content',
 'logFee',
 'polarity',
 'Subjectivity',
 'state_gdp',
 'state_population',
 'gdp_vs_population']

In [10]:

#Defino features a usar en un primer modelo de prueba
features = ['Type',
 'Age',
 'Breed1',
 'Breed2',
 'Gender',
 'Color1',
 'Color2',
 'Color3',
 'MaturitySize',
 'FurLength',
 'Vaccinated',
 'Dewormed',
 'Sterilized',
 'Health',
 'Quantity',
 'Fee',
 'State',
 'VideoAmt',
 'PhotoAmt',
 'HasName',
 'DescLen',
 'score_salud',
 'Puro',
 'AgeRange',
 'Color_comb',
 'Content',
 'logFee',
 'polarity',
 'Subjectivity',
 'state_gdp',
 'state_population',
 'gdp_vs_population'
 ]

label = 'AdoptionSpeed'

In [11]:
#Genero dataframes de train y test con sus respectivos targets
X_train = train[features]
y_train = train[label]

X_test = test[features]
y_test = test[label]

## Modelo con cross validation y conjunto de test

In [12]:
#Genero una metrica para que lightGBM haga la evaluación y pueda hacer early_stopping en el cross validation
def lgb_custom_metric_kappa(dy_pred, dy_true):
    metric_name = 'kappa'
    value = cohen_kappa_score(dy_true.get_label(),dy_pred.argmax(axis=1),weights = 'quadratic')
    is_higher_better = True
    return(metric_name, value, is_higher_better)

#Funcion objetivo a optimizar. En este caso vamos a hacer 5fold cv sobre el conjunto de train. 
# El score de CV es el objetivo a optimizar. Ademas vamos a usar los 5 modelos del CV para estimar el conjunto de test,
# registraremos en optuna las predicciones, matriz de confusion y el score en test.
# CV Score -> Se usa para determinar el rendimiento de los hiperparametros con precision 
# Test Score -> Nos permite testear que esta todo OK, no use (ni debo usar) esos datos para nada en el entrenamiento 
# o la optimizacion de hiperparametros

def cv_es_lgb_objective(trial):

    #PArametros para LightGBM
    lgb_params = {      
                        #PArametros fijos
                        'objective': 'multiclass',
                        'verbosity':-1,
                        'num_class': len(y_train.unique()),
                        'num_threads': 10,
                        'num_rounds': 2000,
                        #Hiperparametros a optimizar utilizando suggest_float o suggest_int segun el tipo de dato
                        #Se indica el nombre del parametro, valor minimo, valor maximo 
                        #en elgunos casos el parametro log=True para parametros que requieren buscar en esa escala
                        'lambda_l1': trial.suggest_float('lambda_l1', 1e-8, 10.0, log=True),
                        'lambda_l2': trial.suggest_float('lambda_l2', 1e-8, 10.0, log=True),
                        'num_leaves': trial.suggest_int('num_leaves', 2, 300),
                        'feature_fraction': trial.suggest_float('feature_fraction', 0.4, 1.0),
                        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.4, 1.0),
                        'bagging_freq': trial.suggest_int('bagging_freq', 1, 15),
                        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
                        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1)
                        }

    #Voy a generar estimaciones de los 5 modelos del CV sobre los datos test y los acumulo en la matriz scores_ensemble
    scores_ensemble = np.zeros((len(y_test),len(y_train.unique())))

    #Score del 5 fold CV inicializado en 0
    score_folds = 0

    #Numero de splits del CV
    n_splits = 10

    #Objeto para hacer el split estratificado de CV
    skf = StratifiedKFold(n_splits=n_splits)

    for i, (if_index, oof_index) in enumerate(skf.split(X_train, y_train)):
        
        #Dataset in fold (donde entreno) 
        lgb_if_dataset = lgb.Dataset(data=X_train.iloc[if_index],
                                        label=y_train.iloc[if_index],
                                        free_raw_data=False)
        
        #Dataset Out of fold (donde mido la performance del CV)
        lgb_oof_dataset = lgb.Dataset(data=X_train.iloc[oof_index],
                                        label=y_train.iloc[oof_index],
                                        free_raw_data=False)

        #Entreno el modelo
        lgb_model = lgb.train(lgb_params,
                                lgb_if_dataset,
                                valid_sets=lgb_oof_dataset,
                                callbacks=[lgb.early_stopping(30, verbose=False)],
                                feval = lgb_custom_metric_kappa
                                )
        
        #Acumulo los scores (probabilidades) de cada clase para cada uno de los modelos que determino en los folds
        #Se predice el 20% de los datos que separe para tes y no uso para entrenar en ningun fold
        scores_ensemble = scores_ensemble + lgb_model.predict(X_test)
        
        #Score del fold (registros de dataset train que en este fold quedan out of fold)
        score_folds = score_folds + cohen_kappa_score(y_train.iloc[oof_index], 
                                                            lgb_model.predict(X_train.iloc[oof_index]).argmax(axis=1),weights = 'quadratic')/n_splits


    #Guardo prediccion del trial sobre el conjunto de test
    # Genero nombre de archivo
    predicted_filename = os.path.join(PATH_TO_TEMP_FILES,f'test_{trial.study.study_name}_{trial.number}.joblib')
    # Copia del dataset para guardar la prediccion
    predicted_df = test.copy()
    # Genero columna pred con predicciones sumadas de los 5 folds
    predicted_df['pred'] = [scores_ensemble[p,:] for p in range(scores_ensemble.shape[0])]
    # Grabo dataframe en temp_artifacts
    dump(predicted_df, predicted_filename)
    # Indico a optuna que asocie el archivo generado al trial
    upload_artifact(trial, predicted_filename, artifact_store)    

    #Grabo natriz de confusion
    #Nombre de archivo
    cm_filename = os.path.join(PATH_TO_TEMP_FILES,f'cm_{trial.study.study_name}_{trial.number}.jpg')
    #Grabo archivo
    # plot_confusion_matrix(y_test,scores_ensemble.argmax(axis=1)).write_image(cm_filename)
    #Asocio al trial
    # upload_artifact(trial, cm_filename, artifact_store)

    #Determino score en conjunto de test y asocio como metrica adicional en optuna
    test_score = cohen_kappa_score(y_test,scores_ensemble.argmax(axis=1),weights = 'quadratic')
    trial.set_user_attr("test_score", test_score)

    #Devuelvo score del 5fold cv a optuna para que optimice en base a eso
    return(score_folds)

In [ ]:
#Inicio el store de artefactos (archivos) de optuna
artifact_store = FileSystemArtifactStore(base_path=PATH_TO_OPTUNA_ARTIFACTS)

#Genero estudio
study = optuna.create_study(direction='maximize',
                            storage="sqlite:///../work/db.sqlite3",  # Specify the storage URL here.
                            study_name="04 - LGB Multiclass CV - FE12",
                            load_if_exists = True)
#Corro la optimizacion
study.optimize(cv_es_lgb_objective, n_trials=100)

[I 2025-08-21 21:42:48,858] A new study created in RDB with name: TEST3
C:\Users\josek\AppData\Local\Temp\ipykernel_1824\2553792068.py:89: FutureWarning: upload_artifact() got {'file_path', 'artifact_store', 'study_or_trial'} as positional arguments but they were expected to be given as keyword arguments.
Positional arguments ['study_or_trial', 'file_path', 'artifact_store'] in upload_artifact() have been deprecated since v4.0.0. They will be replaced with the corresponding keyword arguments in v6.0.0, so please use the keyword specification instead. See https://github.com/optuna/optuna/releases/tag/v4.0.0 for details.
  upload_artifact(trial, predicted_filename, artifact_store)
[I 2025-08-21 21:42:57,897] Trial 0 finished with value: 0.3816025268620932 and parameters: {'lambda_l1': 2.6182000453923572e-05, 'lambda_l2': 0.02637719742577083, 'num_leaves': 271, 'feature_fraction': 0.5526991765845005, 'bagging_fraction': 0.9431359155010582, 'bagging_freq': 2, 'min_child_samples': 79, 'lear

A continuación se detallan las pruebas hechas, quedándonos con 04 - LGB Multiclass CV - FE12 en base a los resultados de los estudio de optuna

- 04 - LGB Multiclass CV - FE1: Con variables de HasName y DescLen
- 04 - LGB Multiclass CV - FE3: Con variables de HasName , DescLen y score_salud
- 04 - LGB Multiclass CV - FE4: Con variables de HasName , DescLen, score_salud, pureza, rangoedad, cant_colores
- 04 - LGB Multiclass CV - FE5: Con variables de HasName , DescLen, score_salud, pureza, rangoedad, cant_colores, content, lr
- 04 - LGB Multiclass CV - FE6: Con variables de HasName , DescLen, score_salud, pureza, rangoedad, cant_colores, content, lr +bag + 512 num_leaves
- 04 - LGB Multiclass CV - FE7: Con variables de HasName , DescLen, score_salud, pureza, rangoedad, cant_colores, content, lr +bag + 300 num_leaves +10cv
- 04 - LGB Multiclass CV - FE8: Con variables de HasName , DescLen, score_salud, pureza, rangoedad, cant_colores, content, logFee lr +bag + 300 num_leaves +10cv
- 04 - LGB Multiclass CV - FE9: Con variables de HasName , DescLen, score_salud, pureza, rangoedad, cant_colores, content, logFee lr +bag + 300 num_leaves +10cv . TEST_SIZE = .3
- 04 - LGB Multiclass CV - FE10: Con variables de HasName , DescLen, score_salud, pureza, rangoedad, cant_colores, content, logFee lr +bag + 300 num_leaves +10cv + polarity
- 04 - LGB Multiclass CV - FE11: Con variables de HasName , DescLen, score_salud, pureza, rangoedad, cant_colores, content, logFee lr +bag15 + 300 num_leaves +10cv + polarity + subjectivity
- 04 - LGB Multiclass CV - FE12: Con variables de HasName , DescLen, score_salud, pureza, rangoedad, cant_colores, content, logFee lr +bag15 + 300 num_leaves +10cv + polarity(completo) + subjectivity + GPD


